# gemm-opt: CUDA stage on Colab

Runtime → Change runtime type → **T4 GPU**. Run top to bottom.

Dev loop: edit kernels on your machine → `git push` → rerun the **Get code** and **Build** cells here.

In [ ]:
!nvidia-smi

## Get code

In [ ]:
import os
if not os.path.exists('/content/gemm-opt'):
    !git clone https://github.com/carolynl950/gemm-opt.git /content/gemm-opt
%cd /content/gemm-opt
!git pull && git log --oneline -1

## Build (CMake ≥ 3.25 + Ninja; CUDA arch detected from the GPU)

In [ ]:
!pip install -q "cmake>=3.25" ninja
!export PATH=/usr/local/cuda/bin:$PATH && cmake --preset release 2>&1 | grep -E 'CUDA|OpenMP|BLAS|Error' ; cmake --build --preset release 2>&1 | tail -20

## Correctness: every CUDA kernel vs CPU naive

In [ ]:
!./build/release/test_cuda

## Benchmarks (only after test_cuda passes)

In [ ]:
!mkdir -p results
!./build/release/bench_cuda --reps 10 512 1024 2048 4096 > results/stage5_square_t4.csv
!head -3 results/stage5_square_t4.csv

In [ ]:
shapes = lambda T: f'--shape {T}x2304x768 --shape {T}x768x768 --shape {T}x3072x768 --shape {T}x768x3072'
!./build/release/bench_cuda --reps 20 {shapes(512)} > results/stage5_prefill_t4.csv
!./build/release/bench_cuda --reps 100 --cold {shapes(1)} > results/stage5_decode_cold_t4.csv

In [ ]:
import pandas as pd
for f in ['stage5_square_t4', 'stage5_prefill_t4', 'stage5_decode_cold_t4']:
    df = pd.read_csv(f'results/{f}.csv')
    print(f)
    print(df.groupby(['implementation', 'm', 'n', 'k'])['gflops'].median().unstack(0).round(1), '\n')

## Download the CSVs (commit them to `results/` on your machine)

In [ ]:
from google.colab import files
for f in ['stage5_square_t4', 'stage5_prefill_t4', 'stage5_decode_cold_t4']:
    files.download(f'results/{f}.csv')